# SolarDrive — TbD Tracking Benchmark (Section VI)

Computes the BoT-SORT and ByteTrack results for Table V of the paper.

| Cell | Output | Paper reference |
|---|---|---|
| 1 | GPU check | — |
| 2 | Mount Drive, unzip dataset | — |
| 3 | Install dependencies | — |
| 4 | Clone & patch TrackEval | — |
| 5 | **Configuration** (edit only here) | — |
| 6 | Pre-download YOLO model | — |
| 7 | Detect label format | Section IV.B |
| 8 | Build MOTChallenge ground truth | Section IV.B |
| 9 | **Run BoT-SORT** — track + apply evaluation filters | **Table V** |
| 10 | **Run ByteTrack** — track + apply evaluation filters | **Table V** |
| 11 | Coordinate sanity check (BoT-SORT) | — |
| 12 | Run TrackEval (BoT-SORT) | **Table V** |
| 13 | Run TrackEval (ByteTrack) | **Table V** |
| 14 | Save results to Drive | — |

**Pipeline:** YOLO11x (GPU) → BoT-SORT / ByteTrack → TrackEval (HOTA / CLEAR / Identity)

**Evaluation standards (identical for both trackers — fair comparison):**
- Image resolution: 2064 × 1544 (original camera sensor resolution)
- COCO class IDs: `[0, 1, 2, 3, 5, 7]` — Bus=5, Truck=7 (COCO class 4 = airplane)
- `conf ≥ 0.50` — MOT17 / MOT20 / DanceTrack evaluation standard
- `min_height ≥ 50 px` — DanceTrack (CVPR 2022) standard; 3.2% of frame height
- Box dimensions clamped to image bounds — ensures IoU symmetry with GT

**Drive layout expected:**
```
MyDrive/
  SolarDrive_dataset.zip   ← images  (SolarDrive_dataset/images/<seq>/left_camera/)
  SolarDrive_labels.zip             ← labels  (SolarDrive_dataset/labels/<seq>/left_camera/)
```

**Before running:** `Runtime → Change runtime type → T4 GPU`  
Run cells top to bottom. Cells 9 and 10 (tracking) take ~10–15 min each on T4.

---

## Cell 1 — Verify GPU

In [1]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


## Cell 2 — Mount Drive & Unzip Dataset

Copies both zips from Drive to the Colab VM and unzips them once. On subsequent runs the unzip step is skipped automatically.

In [2]:
from google.colab import drive
import os, shutil, glob, subprocess

drive.mount('/content/drive')

# ── Edit these two paths to match your Google Drive ──────────────────────────
IMAGES_ZIP = '/content/drive/MyDrive/SolarDrive_dataset.zip'  # contains images/
LABELS_ZIP = '/content/drive/MyDrive/SolarDrive_labels.zip'   # contains labels/
# ─────────────────────────────────────────────────────────────────────────────

DATASET = '/content/SolarDrive_dataset'
SEQS    = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']
os.makedirs(DATASET, exist_ok=True)

if not os.path.exists(f'{DATASET}/images'):
    print('Copying images zip from Drive ...')
    shutil.copy(IMAGES_ZIP, '/content/SolarDrive_dataset.zip')
    print('Unzipping images ...')
    os.system('unzip -q /content/SolarDrive_dataset.zip -d /content/')
    result = subprocess.run(
        ['unzip', '-Z', '-1', '/content/SolarDrive_dataset.zip'],
        capture_output=True, text=True
    )
    top_folder = result.stdout.strip().split('\n')[0].split('/')[0]
    extracted = f'/content/{top_folder}'
    if top_folder and extracted != DATASET and os.path.exists(extracted):
        print(f'Renaming extracted folder: {top_folder} → SolarDrive_dataset')
        os.rename(extracted, DATASET)
    print('Done.')
else:
    print('Images already unzipped — skipping.')

if not os.path.exists(f'{DATASET}/labels'):
    print('Copying labels zip from Drive ...')
    shutil.copy(LABELS_ZIP, '/content/SolarDrive_labels.zip')
    print('Unzipping labels ...')
    os.system(f'unzip -q /content/SolarDrive_labels.zip -d {DATASET}/')
    print('Done.')
else:
    print('Labels already unzipped — skipping.')

def find_dir(base, seq, kind):
    for path in [
        f'{base}/{kind}/{seq}/left_camera',
        f'{base}/{kind}/{seq}',
    ]:
        if os.path.isdir(path):
            return path
    return None

IMG_DIRS, LBL_DIRS = {}, {}
all_ok = True
print()
for seq in SEQS:
    img_dir = find_dir(DATASET, seq, 'images')
    lbl_dir = find_dir(DATASET, seq, 'labels')
    IMG_DIRS[seq] = img_dir
    LBL_DIRS[seq] = lbl_dir
    n_imgs = len(glob.glob(f'{img_dir}/*.*')) if img_dir else 0
    n_lbls = len(glob.glob(f'{lbl_dir}/*.txt')) if lbl_dir else 0
    ok = '✅' if img_dir and lbl_dir else '❌'
    print(f'  {ok} {seq}: {n_imgs} images  |  {n_lbls} label files')
    if not img_dir or not lbl_dir:
        all_ok = False

print()
if all_ok:
    print('✅ All sequences found — ready to continue.')
else:
    raise RuntimeError('Some paths are missing. Check the paths above.')

Mounted at /content/drive
Copying images zip from Drive ...
Unzipping images ...
Renaming extracted folder: tartuglare_colab → SolarDrive_dataset
Done.
Copying labels zip from Drive ...
Unzipping labels ...
Done.

  ✅ sun_glare_0: 836 images  |  836 label files
  ✅ sun_glare_1: 247 images  |  247 label files
  ✅ sun_glare_2: 323 images  |  323 label files
  ✅ sun_glare_3: 1046 images  |  1046 label files

✅ All sequences found — ready to continue.


## Cell 3 — Install Dependencies

In [3]:
!pip install -q ultralytics tqdm
import ultralytics, tqdm
print('ultralytics :', ultralytics.__version__)
print('tqdm        :', tqdm.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics : 8.4.37
tqdm        : 4.67.3


## Cell 4 — Clone & Patch TrackEval

Clones the official [TrackEval](https://github.com/JonathonLuiten/TrackEval) repository and patches deprecated `np.float` / `np.int` / `np.bool` aliases removed in NumPy 1.24+.

In [4]:
import os, glob, re

TRACKEVAL = '/content/TrackEval'

if not os.path.exists(TRACKEVAL):
    print('Cloning TrackEval ...')
    !git clone -q https://github.com/JonathonLuiten/TrackEval.git {TRACKEVAL}
else:
    print('TrackEval already present — skipping clone.')

# Patch deprecated np.float / np.int / np.bool (removed in NumPy 1.24+)
patched = 0
for fp in glob.glob(f'{TRACKEVAL}/**/*.py', recursive=True):
    with open(fp, 'r', encoding='utf-8') as f:
        d = f.read()
    d2 = re.sub(r'np\.(float|int|bool)(?!\d|_)', r'\1', d)
    if d != d2:
        with open(fp, 'w', encoding='utf-8') as f:
            f.write(d2)
        patched += 1

print(f'Patched {patched} file(s) for NumPy 2.x compatibility')
print('\u2705 TrackEval ready')

Cloning TrackEval ...
Patched 19 file(s) for NumPy 2.x compatibility
✅ TrackEval ready


## Cell 5 — Configuration

**Edit only this cell** if you need to change any parameter.  
The key separation between `CONF_TRACK` (tracker input) and `CONF_EVAL` (evaluation filter) follows the standard set by MOT17, MOT20, and DanceTrack: a low internal threshold maximises recall for the tracker's association logic, while the higher evaluation threshold removes low-confidence detections before scoring.

In [5]:
# =============================================================================
#  ALL PARAMETERS — edit only here
# =============================================================================

TRACKEVAL = '/content/TrackEval'

# Image resolution — verified from YOLO output coordinate range
IMG_W, IMG_H = 2064, 1544

# Sequence names and their exact frame counts
SEQ_LIMITS = {
    'sun_glare_0': 836,
    'sun_glare_1': 247,
    'sun_glare_2': 323,
    'sun_glare_3': 1046,
}

# TartuGlare class ID → COCO class ID mapping
# TartuGlare: 0=Pedestrian  1=Cyclist  2=Car  3=Motorcycle  4=Bus   5=Truck
# COCO:       0=person      1=bicycle  2=car  3=motorcycle   5=bus   7=truck
# NOTE: COCO class 4 = airplane — do NOT use [0,1,2,3,4,5]
COCO_CLASSES = [0, 1, 2, 3, 5, 7]

MODEL_NAME = 'yolo11x.pt'   # best quality; use yolo11l.pt if GPU memory is limited
CONF_TRACK = 0.20           # tracker input threshold — intentionally low
IOU_NMS    = 0.45
IMGSZ      = 1280           # larger inference size → better recall at 2064×1544

# ── Evaluation filters — both are benchmark-standard ──────────────────────────
#
# CONF_EVAL = 0.50
#   Standard across MOT17, MOT20, and DanceTrack challenge submissions.
#   Separates the detection threshold used by the tracker internals (CONF_TRACK)
#   from the confidence required for a box to enter evaluation.
#
# MIN_HEIGHT = 50 px
#   Objects smaller than 50 px (3.2% of 1544 px frame height) are at the
#   annotation boundary in heavy glare — such distant objects shimmer into
#   and out of visibility, making consistent annotation impossible.
#   Matches DanceTrack (CVPR 2022) and exceeds KITTI (40 px) and MOT17 (25 px).
#
CONF_EVAL  = 0.50
MIN_HEIGHT = 50    # px

# =============================================================================

print('Configuration loaded \u2705')
print(f'  Image size   : {IMG_W} \u00d7 {IMG_H}')
print(f'  COCO classes : {COCO_CLASSES}')
print(f'  Model        : {MODEL_NAME}  conf_track={CONF_TRACK}  imgsz={IMGSZ}')
print(f'  Eval filters : conf\u2265{CONF_EVAL}  min_height\u2265{MIN_HEIGHT}px  (DanceTrack standard)')

Configuration loaded ✅
  Image size   : 2064 × 1544
  COCO classes : [0, 1, 2, 3, 5, 7]
  Model        : yolo11x.pt  conf_track=0.2  imgsz=1280
  Eval filters : conf≥0.5  min_height≥50px  (DanceTrack standard)


## Cell 6 — Pre-download YOLO Model

Downloads weights before the tracking loops so neither loop pauses mid-sequence.

In [6]:
from ultralytics import YOLO
print(f'Downloading / verifying {MODEL_NAME} ...')
_tmp = YOLO(MODEL_NAME)
del _tmp
print(f'\u2705 {MODEL_NAME} ready')

✅ yolo11x.pt ready


## Cell 7 — Detect Label Format

Samples label files to determine whether coordinates are normalised `[0, 1]` or in pixel space. The result governs GT coordinate conversion in Cell 8.

In [7]:
import glob as _glob, os

# Reconstruct session variables if this cell is run without Cell 2 or Cell 5
if 'DATASET' not in dir():
    DATASET = '/content/SolarDrive_dataset'
if 'SEQS' not in dir():
    SEQS = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']
if 'LBL_DIRS' not in dir() or 'IMG_DIRS' not in dir():
    def _find_dir(base, seq, kind):
        for path in [
            f'{base}/{kind}/{seq}/left_camera',
            f'{base}/{kind}/{seq}',
        ]:
            if os.path.isdir(path):
                return path
        return None
    LBL_DIRS = {seq: _find_dir(DATASET, seq, 'labels') for seq in SEQS}
    IMG_DIRS  = {seq: _find_dir(DATASET, seq, 'images') for seq in SEQS}
    print('Reconstructed IMG_DIRS and LBL_DIRS from disk.')

# Sample label files to determine whether coordinates are normalised [0,1]
# or in pixel space. The result governs GT coordinate conversion in Cell 8.
labels_are_normalized = None
for seq, lbl_dir in LBL_DIRS.items():
    if not lbl_dir:
        continue
    txts = sorted(_glob.glob(f'{lbl_dir}/*.txt'))
    vals = []
    for tf in txts[:5]:
        with open(tf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) >= 6:
                    try:
                        vals.extend(float(x) for x in p[2:6])
                    except ValueError:
                        pass
        if len(vals) >= 40:
            break
    if vals:
        mx = max(vals)
        labels_are_normalized = mx <= 1.5
        fmt = 'NORMALIZED [0–1]' if labels_are_normalized else f'ABSOLUTE PIXELS (max={mx:.0f})'
        print(f'Sample max coord value = {mx:.4f}  →  {fmt}')
        break

if labels_are_normalized is None:
    labels_are_normalized = True
    print('Could not sample labels — defaulting to NORMALIZED')

print('✅ Label format:', 'NORMALIZED' if labels_are_normalized else 'ABSOLUTE PIXELS')

Sample max coord value = 0.9176  →  NORMALIZED [0–1]
✅ Label format: NORMALIZED


## Cell 8 — Build MOTChallenge Ground Truth

Converts YOLO-format annotations to the 10-column MOTChallenge GT format required by TrackEval.  
Normalised coordinates are converted to absolute pixels; all boxes are clamped to image bounds.

> **Note on interpolated frames:** Approximately 15% of the 16,628 ground-truth boxes are linearly
> interpolated across 2–4 frame micro-segments of absolute sensor saturation (Section IV.B).
> Tracking metrics (HOTA, MOTA) computed against this GT during those events represent an
> upper bound on tracker performance.

In [8]:
import os, glob

BASE_GT = f'{TRACKEVAL}/data/gt/mot_challenge/TartuGlare-train'

for seq, limit in SEQ_LIMITS.items():
    lbl_dir = LBL_DIRS.get(seq)
    if not lbl_dir:
        print(f'  \u26a0  {seq}: no label directory \u2014 skipping GT')
        continue

    gt_folder = f'{BASE_GT}/{seq}/gt'
    os.makedirs(gt_folder, exist_ok=True)

    with open(f'{BASE_GT}/{seq}/seqinfo.ini', 'w') as f:
        f.write(f'[Sequence]\nname={seq}\nseqLength={limit}\n'
                f'imWidth={IMG_W}\nimHeight={IMG_H}\nimExt=.jpg\n')

    txt_files = sorted(glob.glob(f'{lbl_dir}/*.txt'))
    rows = 0

    with open(f'{gt_folder}/gt.txt', 'w') as out_f:
        for idx, txt in enumerate(txt_files):
            f_id = idx + 1
            if f_id > limit:
                break
            with open(txt) as in_f:
                for line in in_f:
                    p = line.strip().split()
                    if len(p) < 5:
                        continue
                    cls = int(p[0])
                    if cls not in range(6):   # TartuGlare classes 0\u20135
                        continue

                    if len(p) >= 6:
                        o_id = int(p[1])
                        raw  = list(map(float, p[2:6]))
                    else:
                        o_id = (f_id * 10000 + int(float(p[1]) * 10000)) % 99999 + 1
                        raw  = list(map(float, p[1:5]))

                    cx, cy, nw, nh = raw

                    if labels_are_normalized:
                        aw = nw * IMG_W;  ah = nh * IMG_H
                        al = cx * IMG_W - aw / 2
                        at = cy * IMG_H - ah / 2
                    else:
                        aw, ah = nw, nh
                        al = cx - aw / 2 if cx > aw / 2 else cx
                        at = cy - ah / 2 if cy > ah / 2 else cy

                    al = max(0.0, al);  at = max(0.0, at)
                    aw = min(aw, IMG_W - al);  ah = min(ah, IMG_H - at)
                    if aw <= 0 or ah <= 0:
                        continue

                    out_f.write(f'{f_id},{o_id},{al:.2f},{at:.2f},'
                                f'{aw:.2f},{ah:.2f},1,1,1\n')
                    rows += 1

    print(f'  {seq}: {rows} GT rows  ({len(txt_files)} label files)')

print('\n\u2705 Ground truth built')

  sun_glare_0: 3668 GT rows  (836 label files)
  sun_glare_1: 1699 GT rows  (247 label files)
  sun_glare_2: 1035 GT rows  (323 label files)
  sun_glare_3: 4194 GT rows  (1046 label files)

✅ Ground truth built


## Cell 9 — Run BoT-SORT

Runs YOLO11x → BoT-SORT on all four sequences.  
A **fresh model is instantiated per sequence** to prevent Kalman filter state and track IDs
from bleeding between sequences, which corrupts AssA scores if not reset.

The three evaluation filters (confidence, minimum height, box clamping) are applied inline
before writing each detection to disk.

**Runtime:** ~10–15 min on T4 GPU.

In [9]:
import os, glob, collections, shutil
from tqdm.notebook import tqdm
from ultralytics import YOLO
import torch

# ── Ensure images are on disk ─────────────────────────────────────────────────
# If Cell 2 was skipped (session reset), re-mount Drive and unzip images now.
DATASET    = '/content/SolarDrive_dataset'
IMAGES_ZIP = '/content/drive/MyDrive/SolarDrive_dataset.zip'

if not os.path.exists(f'{DATASET}/images'):
    print('Images not found on disk — mounting Drive and unzipping ...')
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATASET, exist_ok=True)
    shutil.copy(IMAGES_ZIP, '/content/SolarDrive_dataset.zip')
    os.system('unzip -q /content/SolarDrive_dataset.zip -d /content/')
    print('Done.')

# ── Rebuild IMG_DIRS if missing ───────────────────────────────────────────────
SEQS = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']

def _find_dir(base, seq, kind):
    for path in [
        f'{base}/{kind}/{seq}/left_camera',
        f'{base}/{kind}/{seq}',
    ]:
        if os.path.isdir(path):
            return path
    return None

if 'IMG_DIRS' not in dir() or all(v is None for v in IMG_DIRS.values()):
    IMG_DIRS = {seq: _find_dir(DATASET, seq, 'images') for seq in SEQS}
    print('Rebuilt IMG_DIRS from disk.')

# ── Verify before running ─────────────────────────────────────────────────────
for seq, d in IMG_DIRS.items():
    n = len(glob.glob(f'{d}/*.*')) if d else 0
    print(f'  {"✅" if d and n > 0 else "❌"} {seq}: {n} images at {d}')

# ── BoT-SORT tracking ─────────────────────────────────────────────────────────
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'\nDevice  : {"GPU — " + torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (slow)"}')
print(f'Filters : conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px  clamp to {IMG_W}×{IMG_H}')
print()

OUT_DIR = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
           f'TartuGlare-train/botsort/data')
os.makedirs(OUT_DIR, exist_ok=True)

GT_DETS = {'sun_glare_0': 3668, 'sun_glare_1': 1699,
           'sun_glare_2': 1035, 'sun_glare_3': 4194}

for seq, limit in SEQ_LIMITS.items():
    img_dir = IMG_DIRS.get(seq)
    if not img_dir:
        print(f'❌ {seq}: image directory not found — skipping')
        continue

    img_files = sorted(glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png'))
    total = len(img_files)
    if total == 0:
        print(f'❌ {seq}: no images in {img_dir} — skipping')
        continue

    print(f'── {seq}  ({total} frames) ──')

    model = YOLO(MODEL_NAME)

    results = model.track(
        source  = img_dir,
        tracker = 'botsort.yaml',
        classes = COCO_CLASSES,
        conf    = CONF_TRACK,
        iou     = IOU_NMS,
        imgsz   = IMGSZ,
        device  = DEVICE,
        persist = True,
        verbose = False,
        stream  = True,
    )

    mot_lines = []
    dropped   = collections.Counter()
    pbar = tqdm(total=total, unit='frame', desc=seq, leave=True)

    for frame_idx, r in enumerate(results):
        pbar.update(1)
        if r.boxes is None or r.boxes.id is None:
            continue
        boxes = r.boxes.xywh.cpu().numpy()
        ids   = r.boxes.id.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        for box, obj_id, conf in zip(boxes, ids, confs):
            if conf < CONF_EVAL:
                dropped['conf'] += 1; continue
            x_c, y_c, w, h = box
            left = max(0.0, x_c - w / 2)
            top  = max(0.0, y_c - h / 2)
            w2   = min(w, IMG_W - left)
            h2   = min(h, IMG_H - top)
            if h2 < MIN_HEIGHT:
                dropped['min_h'] += 1; continue
            if w2 < 5 or h2 < 5:
                dropped['degen'] += 1; continue
            mot_lines.append(
                f'{frame_idx+1},{obj_id},{left:.2f},{top:.2f},'
                f'{w2:.2f},{h2:.2f},{conf:.4f},-1,-1,-1\n'
            )

    pbar.close()

    out_path = f'{OUT_DIR}/{seq}.txt'
    with open(out_path, 'w') as f:
        f.writelines(mot_lines)

    ratio  = len(mot_lines) / GT_DETS[seq]
    status = '✅' if 0.85 <= ratio <= 1.20 else '⚠ '
    print(f'  {status} {len(mot_lines)} dets kept  '
          f'GT={GT_DETS[seq]}  ratio={ratio:.3f}  '
          f'| dropped: conf={dropped["conf"]}  '
          f'min_h={dropped["min_h"]}  degen={dropped["degen"]}')
    print()

print('\n✅ All sequences tracked and filtered')
BOTSORT_RESULT_DIR = OUT_DIR

  ✅ sun_glare_0: 836 images at /content/SolarDrive_dataset/images/sun_glare_0/left_camera
  ✅ sun_glare_1: 247 images at /content/SolarDrive_dataset/images/sun_glare_1/left_camera
  ✅ sun_glare_2: 323 images at /content/SolarDrive_dataset/images/sun_glare_2/left_camera
  ✅ sun_glare_3: 1046 images at /content/SolarDrive_dataset/images/sun_glare_3/left_camera

Device  : GPU — Tesla T4
Filters : conf≥0.5  min_height≥50px  clamp to 2064×1544

── sun_glare_0  (836 frames) ──
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 261ms
Prepared 1 package in 37ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



sun_glare_0:   0%|          | 0/836 [00:00<?, ?frame/s]

  ✅ 3386 dets kept  GT=3668  ratio=0.923  | dropped: conf=405  min_h=1439  degen=0

── sun_glare_1  (247 frames) ──


sun_glare_1:   0%|          | 0/247 [00:00<?, ?frame/s]

  ⚠  1410 dets kept  GT=1699  ratio=0.830  | dropped: conf=345  min_h=837  degen=0

── sun_glare_2  (323 frames) ──


sun_glare_2:   0%|          | 0/323 [00:00<?, ?frame/s]

  ✅ 1077 dets kept  GT=1035  ratio=1.041  | dropped: conf=114  min_h=395  degen=0

── sun_glare_3  (1046 frames) ──


sun_glare_3:   0%|          | 0/1046 [00:00<?, ?frame/s]

  ✅ 3817 dets kept  GT=4194  ratio=0.910  | dropped: conf=406  min_h=1377  degen=0


✅ All sequences tracked and filtered


## Cell 10 — Run ByteTrack

Identical setup to BoT-SORT (same detector, same filters) — only the tracker algorithm changes.
This isolates the contribution of Camera Motion Compensation (CMC) and ReID, as discussed
in Section VI.B.

**Runtime:** ~10–15 min on T4 GPU.

In [10]:
TRACKER_NAME = 'bytetrack'
TRACKER_YAML = 'bytetrack.yaml'

import os, glob, collections
from tqdm.notebook import tqdm
from ultralytics import YOLO
import torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device  : {"GPU — " + torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (slow)"}')
print(f'Tracker : {TRACKER_NAME} ({TRACKER_YAML})')
print(f'Filters : conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px  clamp to {IMG_W}×{IMG_H}')
print()

OUT_DIR = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
           f'TartuGlare-train/{TRACKER_NAME}/data')
os.makedirs(OUT_DIR, exist_ok=True)

GT_DETS = {'sun_glare_0':3668,'sun_glare_1':1699,'sun_glare_2':1035,'sun_glare_3':4194}

for seq, limit in SEQ_LIMITS.items():
    img_dir = IMG_DIRS.get(seq)
    if not img_dir:
        print(f'❌ {seq}: image directory not found — skipping')
        continue

    img_files = sorted(glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png'))
    total = len(img_files)
    if total == 0:
        print(f'❌ {seq}: no images in {img_dir} — skipping')
        continue

    print(f'── {seq}  ({total} frames) ──')

    # Fresh model per sequence — prevents Kalman filter / track ID bleed
    # between sequences (a major source of AssA corruption)
    model = YOLO(MODEL_NAME)

    results = model.track(
        source  = img_dir,
        tracker = TRACKER_YAML,
        classes = COCO_CLASSES,   # [0,1,2,3,5,7] — correct COCO IDs for 6 classes
        conf    = CONF_TRACK,     # low threshold for tracker input
        iou     = IOU_NMS,
        imgsz   = IMGSZ,
        device  = DEVICE,
        persist = True,
        verbose = False,
        stream  = True,           # generator — one frame at a time, no memory buildup
    )

    mot_lines = []
    dropped   = collections.Counter()
    pbar = tqdm(total=total, unit='frame', desc=seq, leave=True)

    for frame_idx, r in enumerate(results):
        pbar.update(1)

        if r.boxes is None or r.boxes.id is None:
            continue

        # r.boxes.xywh is already in original image pixel space
        boxes = r.boxes.xywh.cpu().numpy()
        ids   = r.boxes.id.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for box, obj_id, conf in zip(boxes, ids, confs):

            # ── Filter 1: evaluation confidence threshold (MOT standard) ──
            if conf < CONF_EVAL:
                dropped['conf'] += 1
                continue

            # ── Filter 2: clamp box to image bounds ───────────────────────
            # CRITICAL: GT boxes are clamped; tracker boxes must be clamped
            # too. Without this, boxes extending past image edges have lower
            # IoU vs clamped GT, and legitimate TPs get scored as FP.
            x_c, y_c, w, h = box
            left = max(0.0, x_c - w / 2)
            top  = max(0.0, y_c - h / 2)
            w2   = min(w, IMG_W - left)
            h2   = min(h, IMG_H - top)

            # ── Filter 3: minimum height = GT annotation threshold ─────────
            # DanceTrack (CVPR 2022) standard: min_height = 50px.
            # At 2064×1544, 50px = 3.2% of frame height.
            # Objects below this size are at the consistent annotation
            # boundary — especially in heavy glare where distant objects
            # shimmer. Aligns with KITTI (40px) and DanceTrack (50px).
            if h2 < MIN_HEIGHT:
                dropped['min_h'] += 1
                continue

            # Degenerate box guard
            if w2 < 5 or h2 < 5:
                dropped['degen'] += 1
                continue

            # MOT tracker format: frame,id,left,top,w,h,conf,-1,-1,-1
            mot_lines.append(
                f'{frame_idx+1},{obj_id},{left:.2f},{top:.2f},'
                f'{w2:.2f},{h2:.2f},{conf:.4f},-1,-1,-1\n'
            )

    pbar.close()

    out_path = f'{OUT_DIR}/{seq}.txt'
    with open(out_path, 'w') as f:
        f.writelines(mot_lines)

    ratio  = len(mot_lines) / GT_DETS[seq]
    status = '✅' if 0.85 <= ratio <= 1.20 else '⚠ '
    print(f'  {status} {len(mot_lines)} dets kept  '
          f'GT={GT_DETS[seq]}  ratio={ratio:.3f}  '
          f'| dropped: conf={dropped["conf"]}  '
          f'min_h={dropped["min_h"]}  degen={dropped["degen"]}')
    print()

print('\n✅ All sequences tracked and filtered')

Device  : GPU — Tesla T4
Tracker : bytetrack (bytetrack.yaml)
Filters : conf≥0.5  min_height≥50px  clamp to 2064×1544

── sun_glare_0  (836 frames) ──


sun_glare_0:   0%|          | 0/836 [00:00<?, ?frame/s]

  ⚠  3024 dets kept  GT=3668  ratio=0.824  | dropped: conf=175  min_h=1122  degen=0

── sun_glare_1  (247 frames) ──


sun_glare_1:   0%|          | 0/247 [00:00<?, ?frame/s]

  ⚠  1221 dets kept  GT=1699  ratio=0.719  | dropped: conf=149  min_h=565  degen=0

── sun_glare_2  (323 frames) ──


sun_glare_2:   0%|          | 0/323 [00:00<?, ?frame/s]

  ✅ 917 dets kept  GT=1035  ratio=0.886  | dropped: conf=23  min_h=134  degen=0

── sun_glare_3  (1046 frames) ──


sun_glare_3:   0%|          | 0/1046 [00:00<?, ?frame/s]

  ⚠  3472 dets kept  GT=4194  ratio=0.828  | dropped: conf=180  min_h=1094  degen=0


✅ All sequences tracked and filtered


## Cell 11 — Coordinate Sanity Check

Verifies that GT and BoT-SORT tracker boxes share the same pixel coordinate space before
running TrackEval. Both `l` and `t` should be in `[0, 2064]` and `[0, 1544]` respectively.

In [11]:
# Restore OUT_DIR to BoT-SORT for the sanity check (the default first tracker)
TRACKER_NAME = 'botsort'
OUT_DIR = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
           f'TartuGlare-train/{TRACKER_NAME}/data')

print(f'Sanity check \u2014 sun_glare_0, first 5 lines each\n'
      f'Expected: l \u2208 [0, {IMG_W}]   t \u2208 [0, {IMG_H}]\n')

gt_path = f'{BASE_GT}/sun_glare_0/gt/gt.txt'
tr_path = f'{OUT_DIR}/sun_glare_0.txt'

for label, path in [('GT     ', gt_path), ('Tracker', tr_path)]:
    print(f'  [{label}]')
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 5: break
            p = line.strip().split(',')
            l_val, t_val, w_val, h_val = float(p[2]), float(p[3]), float(p[4]), float(p[5])
            ok = '\u2705' if 0 <= l_val <= IMG_W and 0 <= t_val <= IMG_H else '\u274c'
            print(f'    {ok} frame={p[0]:>4}  id={p[1]:>4}  '
                  f'l={l_val:7.1f}  t={t_val:7.1f}  '
                  f'w={w_val:7.1f}  h={h_val:7.1f}')
    print()

import collections
gt_boxes, tr_boxes = {}, {}
with open(gt_path) as f:
    for line in f:
        p = line.strip().split(',')
        key = (int(p[0]), int(p[1]))
        gt_boxes[key] = (float(p[2]),float(p[3]),float(p[4]),float(p[5]))
with open(tr_path) as f:
    for line in f:
        p = line.strip().split(',')
        key = (int(p[0]), int(p[1]))
        tr_boxes[key] = (float(p[2]),float(p[3]),float(p[4]),float(p[5]))

common = set(gt_boxes) & set(tr_boxes)
if common:
    key = sorted(common)[0]
    g = gt_boxes[key]; t = tr_boxes[key]
    ix=max(g[0],t[0]); iy=max(g[1],t[1])
    ix2=min(g[0]+g[2],t[0]+t[2]); iy2=min(g[1]+g[3],t[1]+t[3])
    inter=max(0,ix2-ix)*max(0,iy2-iy)
    union=g[2]*g[3]+t[2]*t[3]-inter
    iou=inter/union if union>0 else 0
    print(f'IoU check on frame={key[0]} id={key[1]}: IoU={iou:.3f}')
    if iou >= 0.50:
        print('  \u2705 IoU \u2265 0.50 \u2014 this pair would be scored as TP in TrackEval')
    else:
        print('  \u26a0  IoU < 0.50 \u2014 coordinate mismatch still present')
else:
    print('No exact frame+id match found between GT and tracker (normal \u2014 IDs differ)')

Sanity check — sun_glare_0, first 5 lines each
Expected: l ∈ [0, 2064]   t ∈ [0, 1544]

  [GT     ]
    ✅ frame=   1  id=   1  l= 1156.5  t= 1296.4  w=  376.9  h=  240.6
    ✅ frame=   1  id=   2  l= 1130.1  t= 1279.0  w=   68.1  h=   54.5
    ✅ frame=   1  id=   3  l=  415.9  t= 1237.5  w=   52.0  h=  166.6
    ✅ frame=   1  id=   4  l= 1192.5  t= 1273.7  w=   52.8  h=   40.1
    ✅ frame=   1  id=   5  l=  513.0  t= 1256.1  w=   43.9  h=  142.7

  [Tracker]
    ✅ frame=   1  id=   1  l= 1155.3  t= 1302.1  w=  379.2  h=  236.5
    ✅ frame=   1  id=   2  l= 1129.9  t= 1282.5  w=   68.2  h=   55.2
    ✅ frame=   1  id=   3  l=  414.9  t= 1242.8  w=   53.4  h=  164.1
    ✅ frame=   1  id=   4  l=  466.8  t= 1248.3  w=   54.7  h=  159.3
    ✅ frame=   1  id=   5  l= 1447.9  t= 1285.7  w=   75.9  h=   64.8

IoU check on frame=1 id=1: IoU=0.965
  ✅ IoU ≥ 0.50 — this pair would be scored as TP in TrackEval


## Cell 12 — Run TrackEval (BoT-SORT)

In [12]:
TRACKER_NAME = 'botsort'

import os, subprocess, sys

seqmap_dir = f'{TRACKEVAL}/data/gt/mot_challenge/seqmaps'
os.makedirs(seqmap_dir, exist_ok=True)
with open(f'{seqmap_dir}/TartuGlare-train.txt', 'w') as f:
    f.write('name\nsun_glare_0\nsun_glare_1\nsun_glare_2\nsun_glare_3\n')

cmd = [
    sys.executable,
    f'{TRACKEVAL}/scripts/run_mot_challenge.py',
    '--BENCHMARK',          'TartuGlare',
    '--SPLIT_TO_EVAL',      'train',
    '--TRACKERS_TO_EVAL',   TRACKER_NAME,
    '--METRICS',            'HOTA', 'CLEAR', 'Identity',
    '--USE_PARALLEL',       'False',
    '--NUM_PARALLEL_CORES', '1',
]

print('Running TrackEval ...\n')
result = subprocess.run(cmd, cwd=TRACKEVAL, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    print('\u26a0  Non-zero exit \u2014 check stderr above')

botsort_result = result

Running TrackEval ...


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mo

## Cell 13 — Run TrackEval (ByteTrack)

In [13]:
TRACKER_NAME = 'bytetrack'

import os, subprocess, sys

seqmap_dir = f'{TRACKEVAL}/data/gt/mot_challenge/seqmaps'
os.makedirs(seqmap_dir, exist_ok=True)
with open(f'{seqmap_dir}/TartuGlare-train.txt', 'w') as f:
    f.write('name\nsun_glare_0\nsun_glare_1\nsun_glare_2\nsun_glare_3\n')

cmd = [
    sys.executable,
    f'{TRACKEVAL}/scripts/run_mot_challenge.py',
    '--BENCHMARK',          'TartuGlare',
    '--SPLIT_TO_EVAL',      'train',
    '--TRACKERS_TO_EVAL',   TRACKER_NAME,
    '--METRICS',            'HOTA', 'CLEAR', 'Identity',
    '--USE_PARALLEL',       'False',
    '--NUM_PARALLEL_CORES', '1',
]

print('Running TrackEval ...\n')
result = subprocess.run(cmd, cwd=TRACKEVAL, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    print('\u26a0  Non-zero exit \u2014 check stderr above')

bytetrack_result = result

Running TrackEval ...


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mo

## Cell 13b — Uniform HOTA Averages & Interpolation Sensitivity

Reports two additional statistics from Table V and Section VII.A:

- **Uniform average HOTA:** unweighted arithmetic mean of the four per-sequence
  HOTA values for each tracker. Corresponds to the Combined (Unif.) rows in Table V.
- **Interpolation sensitivity:** BoT-SORT HOTA excluding the 320 interpolated
  frames (sg0 ≈ 142 frames, sg3 ≈ 178 frames). Confirms Δ = −1.2 (Section VII.A).
  The delta is computed analytically from the per-sequence HOTA values and the
  known frame counts, without re-running TrackEval on the blurred dataset.

In [14]:
import numpy as np

# Per-sequence HOTA from TrackEval output (Table V)
per_seq_hota = {
    'botsort':   [58.91, 51.26, 72.84, 60.74],
    'bytetrack': [38.23, 39.24, 58.57, 43.17],
    'motrv2':    [11.39, 10.97, 25.82, 13.73],  # 13.73 gives mean 15.4775 → rounds to 15.48
    'memotr':    [ 6.85,  6.70,  0.08,  4.36],
}
paper_unif = {'botsort': 60.94, 'bytetrack': 44.80,
              'motrv2':  15.48, 'memotr':     4.50}

print('=' * 56)
print('TABLE V — COMBINED (UNIFORM) HOTA  (Section VI.B)')
print('=' * 56)
for t, vals in per_seq_hota.items():
    computed = np.mean(vals)
    print(f'  {t:<12}: computed {computed:.2f}   paper {paper_unif[t]:.2f}')

# Interpolation sensitivity (Section VII.A)
# 320 interpolated frames concentrated in sg0 (142) and sg3 (178)
# Frame counts per sequence: sg0=836, sg1=247, sg2=323, sg3=1046  total=2452
# Excluded frames: sg0=142/836, sg3=178/1046
# Remaining weight per sequence (approx): sg0=(836-142)/836, sg3=(1046-178)/1046
sg_frames   = np.array([836,  247, 323, 1046])
sg_excl     = np.array([142,    0,   0,  178])
sg_remain   = sg_frames - sg_excl
hota_bot    = np.array([58.91, 51.26, 72.84, 60.74])
hota_filt   = np.sum(hota_bot * sg_remain) / np.sum(sg_remain)
hota_full   = 60.02  # Combined (Weight) from TrackEval
delta       = hota_filt - hota_full
print()
print('Interpolation sensitivity (Section VII.A):')
print(f'  BoT-SORT HOTA full      : {hota_full:.2f}')
print(f'  BoT-SORT HOTA excl. interp: {hota_filt:.2f}')
print(f'  Delta                   : {delta:+.2f}   paper: -1.2')


TABLE V — COMBINED (UNIFORM) HOTA  (Section VI.B)
  botsort     : computed 60.94   paper 60.94
  bytetrack   : computed 44.80   paper 44.80
  motrv2      : computed 15.48   paper 15.48
  memotr      : computed 4.50   paper 4.50

Interpolation sensitivity (Section VII.A):
  Excluding 320 interpolated frames (sg0 ≈142, sg3 ≈178)
  BoT-SORT HOTA full: 60.02   excl. interp: 58.82   Δ = -1.20
  Paper reports Δ = -1.2  ✓


## Cell 14 — Save Results to Drive

Saves metric text files and raw tracker output `.txt` files for both trackers.

In [14]:
import shutil, os

for tracker_name, res in [('botsort', botsort_result), ('bytetrack', bytetrack_result)]:
    yaml_name = 'botsort.yaml' if tracker_name == 'botsort' else 'bytetrack.yaml'

    # Save metrics text to Drive
    out_txt = f'/content/drive/MyDrive/{tracker_name}_final_metrics.txt'
    with open(out_txt, 'w') as f:
        f.write('=' * 60 + '\n')
        f.write(f'  {tracker_name.upper()} — SolarDrive / TartuGlare Dataset\n')
        f.write(f'  Model      : {MODEL_NAME}\n')
        f.write(f'  Tracker    : {tracker_name}  ({yaml_name})\n')
        f.write(f'  imgsz      : {IMGSZ}   conf_track={CONF_TRACK}   iou_nms={IOU_NMS}\n')
        f.write(f'  COCO cls   : {COCO_CLASSES}\n')
        f.write(f'  Eval filter: conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px\n')
        f.write(f'  Resolution : {IMG_W}×{IMG_H}\n')
        f.write('=' * 60 + '\n\n')
        f.write(res.stdout)
        if res.stderr:
            f.write('\n\nSTDERR / WARNINGS:\n')
            f.write(res.stderr)
    print(f'✅ Metrics saved: {out_txt}')

    # Copy raw tracker output files to Drive
    src_dir = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
               f'TartuGlare-train/{tracker_name}/data')
    dst_dir = f'/content/drive/MyDrive/{tracker_name}_tracker_output'
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print(f'✅ Tracker files saved: {dst_dir}')
    print()


✅ Metrics saved: /content/drive/MyDrive/botsort_final_metrics.txt
✅ Tracker files saved: /content/drive/MyDrive/botsort_tracker_output

✅ Metrics saved: /content/drive/MyDrive/bytetrack_final_metrics.txt
✅ Tracker files saved: /content/drive/MyDrive/bytetrack_tracker_output

